# LazyAd — Phase 1, Spike S3: TripoSR 3D Reconstruction

**Source of Truth:** `Docs/Phase 1, Spike S3 TripoSR 3D Reconstruction.md` & `Docs/PHASE 0 FEASIBILITY & REQUIREMENTS.md` (Approved v2.2)

### Objective
Empirically test whether the user-supplied product image (staged in Spike S2) can be reconstructed into a usable 3D mesh using the **TripoSR** baseline in the active Google Colab runtime (Tesla T4 GPU, Python 3.13, PyTorch 2.11, CUDA 13.0).

### Expected Output
A valid, non-empty GLB model saved at:  
`/content/lazyad_assets/watch_reconstruction.glb`

> **Strict Scope Limits:** Spike S3 is strictly isolated to 3D reconstruction and GLB export/validation. It does **not** install Blender, does **not** generate camera animations, does **not** render advertisements, and does **not** generate audio.

## Step 1: Input Asset Verification & Fallback Upload
Verifies that the S2 product image exists at `/content/lazyad_assets/product_input.png`. If missing (e.g. fresh Colab session), provides an upload fallback.

In [ ]:
import os
import shutil

ASSET_DIR = "/content/lazyad_assets"
os.makedirs(ASSET_DIR, exist_ok=True)
INPUT_IMAGE_PATH = os.path.join(ASSET_DIR, "product_input.png")
GLB_OUTPUT_PATH = os.path.join(ASSET_DIR, "watch_reconstruction.glb")

if os.path.exists(INPUT_IMAGE_PATH) and os.path.getsize(INPUT_IMAGE_PATH) > 0:
    size_kb = os.path.getsize(INPUT_IMAGE_PATH) / 1024.0
    print(f"✅ S2 product input found: {INPUT_IMAGE_PATH} ({size_kb:.2f} KB)")
else:
    print("⚠️ S2 asset not found. Please upload your product image:")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            first_file = list(uploaded.keys())[0]
            shutil.copy2(first_file, INPUT_IMAGE_PATH)
            print(f"✅ Uploaded and staged as: {INPUT_IMAGE_PATH}")
        else:
            raise FileNotFoundError("No image was provided.")
    except ImportError:
        raise FileNotFoundError(f"File not found at {INPUT_IMAGE_PATH} and not running interactively in Colab.")

## Step 2: Environment Inspection & TripoSR Installation
Inspects the active runtime and installs TripoSR dependencies (`trimesh`, `omegaconf`, `einops`, `torchmcubes`). Clones the official repository from `https://github.com/VAST-AI-Research/TripoSR` without altering core PyTorch/CUDA packages.

In [ ]:
import sys
import platform
import torch

print("=" * 72)
print("S3: ENVIRONMENT PRE-CHECK FOR TRIPOSR")
print("=" * 72)
print(f"• Python Version      : {platform.python_version()}")
print(f"• PyTorch Version     : {torch.__version__}")
print(f"• CUDA Available      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"• GPU Model           : {torch.cuda.get_device_name(0)}")
    print(f"• VRAM Capacity       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 72)

# 1. Install build helpers and Python packages required by TripoSR
!pip install -q pybind11 scikit-build-core setuptools
!pip install -q trimesh omegaconf einops transformers

# 2. Install torchmcubes (compiled marching cubes extension)
print("\nInstalling torchmcubes from source...")
!pip install -q --no-build-isolation git+https://github.com/tatsy/torchmcubes.git

# 3. Clone official TripoSR repository if not already present
TRIPOSR_DIR = "/content/TripoSR"
if not os.path.exists(TRIPOSR_DIR):
    print("\nCloning official VAST-AI-Research/TripoSR repository...")
    !git clone -q https://github.com/VAST-AI-Research/TripoSR.git {TRIPOSR_DIR}
    print("✅ Cloned TripoSR.")
else:
    print(f"✅ TripoSR directory already exists at: {TRIPOSR_DIR}")

# Add TripoSR to sys.path so its modules (tsr) can be imported
if TRIPOSR_DIR not in sys.path:
    sys.path.append(TRIPOSR_DIR)

# Verify imports
import trimesh
import torchmcubes
from tsr.system import TSR
print("\n✅ Core TripoSR dependencies and tsr.system successfully imported!")

## Step 3: Transparent-Background-Aware Image Preprocessing
TripoSR expects a **512×512 square RGB image on a neutral grey background (`(128, 128, 128)`)** with the foreground product centered and occupying approximately 85% of the frame.

Since our S2 asset already has an alpha channel:
1. We extract the exact non-transparent bounding box (preserving product proportions).
2. We scale the watch to `foreground_ratio = 0.85` (max dimension 435 px).
3. We composite the watch cleanly onto a 512×512 solid neutral grey canvas.
4. This **completely bypasses redundant rembg models**, preventing edge erosion or AI segmentation artifacts.

In [ ]:
from PIL import Image
import numpy as np
from IPython.display import display

PREPROCESSED_IMAGE_PATH = os.path.join(ASSET_DIR, "watch_preprocessed_512.png")
FOREGROUND_RATIO = 0.85  # Target size of the watch relative to 512px canvas
CANVAS_SIZE = 512
BG_COLOR = (128, 128, 128)  # Neutral grey expected by TripoSR

with Image.open(INPUT_IMAGE_PATH) as raw_img:
    img_rgba = raw_img.convert("RGBA")
    orig_w, orig_h = img_rgba.size
    
    # Get alpha mask
    alpha = img_rgba.split()[-1]
    bbox = alpha.getbbox()
    
    if bbox:
        # Crop tightly to the watch silhouette
        cropped_watch = img_rgba.crop(bbox)
        crop_w, crop_h = cropped_watch.size
        print(f"• Extracted foreground bbox : {bbox} (Dimensions: {crop_w} x {crop_h} px)")
    else:
        # Fallback if no alpha mask found
        cropped_watch = img_rgba
        crop_w, crop_h = orig_w, orig_h
        print("• Notice: Alpha channel is empty or fully opaque. Using full canvas.")
        
    # Scale watch to fit foreground ratio on 512x512 canvas
    target_max_dim = int(CANVAS_SIZE * FOREGROUND_RATIO)
    scale = target_max_dim / max(crop_w, crop_h)
    new_w = int(crop_w * scale)
    new_h = int(crop_h * scale)
    
    resized_watch = cropped_watch.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    # Create solid neutral grey 512x512 canvas
    canvas = Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), BG_COLOR)
    
    # Center the watch
    paste_x = (CANVAS_SIZE - new_w) // 2
    paste_y = (CANVAS_SIZE - new_h) // 2
    
    # Paste using the resized alpha channel as mask
    canvas.paste(resized_watch, (paste_x, paste_y), mask=resized_watch.split()[-1])
    
    # Save preprocessed image
    canvas.save(PREPROCESSED_IMAGE_PATH, format="PNG")

print("=" * 72)
print("S3: PREPROCESSING SUMMARY")
print("=" * 72)
print(f"• Input Resolution    : {orig_w} x {orig_h} px")
print(f"• Preprocessed Output : {PREPROCESSED_IMAGE_PATH} (512 x 512 px, RGB)")
print(f"• Foreground Ratio    : {FOREGROUND_RATIO * 100:.1f}%")
print(f"• Centered Position   : X={paste_x}, Y={paste_y} (Scaled watch size: {new_w}x{new_h})")
print("=" * 72)

print("\nPrepared TripoSR Input Image (512x512 on neutral grey):\n")
display(canvas)

## Step 4: TripoSR Model Inference & 3D Reconstruction
Executes single-image 3D reconstruction on the Tesla T4 GPU:
1. Loads the pretrained TripoSR FP16 model weights from Hugging Face (`stabilityai/TripoSR`).
2. Feeds the preprocessed image through the transformer to produce 3D scene representation codes.
3. Extracts the 3D polygon mesh via Marching Cubes (`mc_resolution = 256`).
4. Records precise step-by-step wall-clock timings and peak VRAM allocation.

In [ ]:
import time
import gc

device = "cuda:0" if torch.cuda.is_available() else "cpu"
MC_RESOLUTION = 256     # Marching Cubes resolution (standard high-detail grid)
CHUNK_SIZE = 8192       # Volume renderer chunk size to keep VRAM consumption bounded

print("=" * 72)
print("S3: TRIPOSR INFERENCE EXECUTION")
print("=" * 72)
print(f"• Target Device         : {device}")
print(f"• Marching Cubes Grid   : {MC_RESOLUTION}^3")
print(f"• Renderer Chunk Size   : {CHUNK_SIZE}")

# Reset peak CUDA stats
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# 1. Model Loading
print("\nLoading TripoSR pretrained model weights...")
t0 = time.time()
model = TSR.from_pretrained(
    "stabilityai/TripoSR",
    config_name="config.yaml",
    weight_name="model.fp16.safetensors",
)
model.renderer.set_chunk_size(CHUNK_SIZE)
model.to(device)
model_load_time = time.time() - t0
print(f"✅ Model loaded in      : {model_load_time:.2f} seconds")

# 2. Image Inference (Feed-Forward)
print("\nRunning neural feed-forward inference...")
preprocessed_img = Image.open(PREPROCESSED_IMAGE_PATH)

t1 = time.time()
with torch.no_grad():
    scene_codes = model(preprocessed_img, device=device)
infer_time = time.time() - t1
print(f"✅ Inference completed in: {infer_time:.2f} seconds")

# 3. Marching Cubes Mesh Extraction
print(f"\nExtracting 3D polygon mesh (resolution={MC_RESOLUTION})...")
t2 = time.time()
with torch.no_grad():
    # extract_mesh returns a list of trimesh.Trimesh objects with vertex colors
    extracted_meshes = model.extract_mesh(scene_codes, True, resolution=MC_RESOLUTION)
mesh_extract_time = time.time() - t2
print(f"✅ Mesh extracted in    : {mesh_extract_time:.2f} seconds")

total_recon_time = infer_time + mesh_extract_time

# VRAM tracking
peak_vram_gb = 0.0
if torch.cuda.is_available():
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

mesh = extracted_meshes[0]

print("-" * 72)
print(f"• Total Reconstruction  : {total_recon_time:.2f} seconds (Inference + Mesh Extraction)")
print(f"• Peak GPU VRAM Used    : {peak_vram_gb:.2f} GB")
print("=" * 72)

## Step 5: Export to Standard GLB & Mesh Integrity Validation
Exports the extracted mesh as a standard binary glTF (`watch_reconstruction.glb`) and verifies geometry with `trimesh`.

In [ ]:
# Export to standard GLB
mesh.export(GLB_OUTPUT_PATH, file_type="glb")

file_exists = os.path.exists(GLB_OUTPUT_PATH)
file_size_mb = os.path.getsize(GLB_OUTPUT_PATH) / (1024 ** 2) if file_exists else 0.0

# Reopen using trimesh to validate geometry independently
reloaded_mesh = None
vertices_count = 0
faces_count = 0
bbox_extents = (0, 0, 0)
is_finite = False
has_vertex_colors = False

if file_exists and file_size_mb > 0:
    loaded = trimesh.load(GLB_OUTPUT_PATH, file_type="glb")
    
    # In trimesh, loading a glb can return a Scene or a Trimesh
    if isinstance(loaded, trimesh.Scene):
        if len(loaded.geometry) > 0:
            reloaded_mesh = list(loaded.geometry.values())[0]
    elif isinstance(loaded, trimesh.Trimesh):
        reloaded_mesh = loaded
        
    if reloaded_mesh is not None:
        vertices_count = len(reloaded_mesh.vertices)
        faces_count = len(reloaded_mesh.faces)
        bbox_extents = tuple(np.round(reloaded_mesh.bounding_box.extents, 4))
        is_finite = bool(np.isfinite(reloaded_mesh.vertices).all())
        has_vertex_colors = hasattr(reloaded_mesh.visual, "vertex_colors") and (reloaded_mesh.visual.vertex_colors is not None)

print("=" * 72)
print("S3: GLB MESH INTEGRITY VALIDATION")
print("=" * 72)
print(f"• Target Output Path   : {GLB_OUTPUT_PATH}")
print(f"• File Size            : {file_size_mb:.2f} MB")
print(f"• Vertex Count         : {vertices_count:,}")
print(f"• Face (Triangle) Count: {faces_count:,}")
print(f"• Bounding Box Extents : X={bbox_extents[0]:.3f}, Y={bbox_extents[1]:.3f}, Z={bbox_extents[2]:.3f}")
print(f"• Finite Geometry Check: {is_finite}")
print(f"• Visual Vertex Colors : {has_vertex_colors}")
print("=" * 72)

## Step 6: Multi-Angle Visual Preview of Reconstructed 3D Watch
Renders 4 orthographic projection angles (Front $0^\circ$, 3/4 Isometric $45^\circ$, Side $90^\circ$, Top $90^\circ$) so the 3D model can be inspected directly inside Colab.

In [ ]:
import matplotlib.pyplot as plt

if reloaded_mesh is not None and vertices_count > 0:
    verts = reloaded_mesh.vertices
    colors = reloaded_mesh.visual.vertex_colors[:, :3] / 255.0 if has_vertex_colors else np.array([0.7, 0.7, 0.7])
    
    # Subsample vertices for fast matplotlib 3D scatter rendering
    num_pts = min(15000, len(verts))
    indices = np.random.choice(len(verts), num_pts, replace=False)
    sample_verts = verts[indices]
    sample_colors = colors[indices] if (colors.ndim > 1 and len(colors) == len(verts)) else colors
    
    fig = plt.figure(figsize=(16, 5))
    
    angles = [
        ("1. Front View (0°)", 0, 0),
        ("2. Three-Quarter Hero (45°)", 20, 45),
        ("3. Side Profile (90°)", 0, 90),
        ("4. Top / Bezel View", 90, 0)
    ]
    
    for i, (title, elev, azim) in enumerate(angles):
        ax = fig.add_subplot(1, 4, i + 1, projection="3d")
        ax.scatter(
            sample_verts[:, 0], 
            sample_verts[:, 2], 
            sample_verts[:, 1], 
            c=sample_colors, 
            s=1.2, 
            alpha=0.8
        )
        ax.view_init(elev=elev, azim=azim)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
        
    plt.suptitle("Reconstructed 3D Watch Mesh — Multi-Angle Inspection", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("❌ No mesh geometry available to render.")

## Step 7: Evaluated Pass / Fail Verdict (S3 Acceptance Criteria)

### Pass Criteria (from `Docs/Phase 1, Spike S3 TripoSR 3D Reconstruction.md`)
1. TripoSR installs and imports cleanly in the active Colab environment.
2. The prepared S2 image is processed and accepted by the model.
3. Single-image 3D reconstruction completes without unhandled errors.
4. A valid GLB file is exported to `/content/lazyad_assets/watch_reconstruction.glb`.
5. The GLB can be reopened and contains non-empty, finite geometry ($>10,000$ vertices and faces).
6. Multi-angle previews are rendered for visual inspection.

In [ ]:
print("=" * 72)
print("SPIKE S3: EVALUATION & ACCEPTANCE VERDICT")
print("=" * 72)

c1_install = "TSR" in locals()
c2_input_accepted = os.path.exists(PREPROCESSED_IMAGE_PATH)
c3_recon_completed = (total_recon_time > 0.0) and (mesh is not None)
c4_glb_exported = file_exists and (file_size_mb > 0.1)
c5_geometry_valid = (vertices_count > 1000) and (faces_count > 1000) and is_finite

print(f"1. TripoSR Installed & Loaded           : [{'PASS' if c1_install else 'FAIL'}]")
print(f"2. Prepared Image Processed & Accepted : [{'PASS' if c2_input_accepted else 'FAIL'}]")
print(f"3. 3D Neural Reconstruction Completed  : [{'PASS' if c3_recon_completed else 'FAIL'}] ({total_recon_time:.2f}s)")
print(f"4. GLB File Exported Successfully      : [{'PASS' if c4_glb_exported else 'FAIL'}] ({file_size_mb:.2f} MB)")
print(f"5. Geometry Non-Empty & Finite         : [{'PASS' if c5_geometry_valid else 'FAIL'}] ({vertices_count:,} verts, {faces_count:,} faces)")

s3_pass = c1_install and c2_input_accepted and c3_recon_completed and c4_glb_exported and c5_geometry_valid

print("-" * 72)
if s3_pass:
    print("🎯 FINAL VERDICT FOR SPIKE S3: [PASS]")
    print(f"   Reconstruction Path: {GLB_OUTPUT_PATH}")
    print(f"   Total Recon Time   : {total_recon_time:.2f}s (Inference: {infer_time:.2f}s, Marching Cubes: {mesh_extract_time:.2f}s)")
    print(f"   Peak VRAM Allocated: {peak_vram_gb:.2f} GB")
    print("\n🔎 VISUAL INSPECTION CHECKLIST FOR BLENDER READINESS (S4):")
    print("   [ ] Is the dial face round/oval and recognizable as a watch?")
    print("   [ ] Does the front hemisphere look visually cohesive?")
    print("   [ ] Is the back melted/deformed (Expected artifact for single-image models)?")
    print("\n👉 Please inspect the multi-angle preview and share your results to confirm S3 Pass!")
else:
    print("❌ FINAL VERDICT FOR SPIKE S3: [FAIL]")
    if not c1_install:
        print("   Failure Reason: TripoSR installation or module import failed.")
    elif not c3_recon_completed:
        print("   Failure Reason: 3D inference or mesh extraction did not complete.")
    elif not c4_glb_exported:
        print("   Failure Reason: GLB file was not created or is empty.")
    elif not c5_geometry_valid:
        print("   Failure Reason: Extracted geometry is degenerate or has fewer than 1000 vertices.")
print("=" * 72)
